In [ ]:
################################################################################
# bitimelygpt - factorial experiment (simulated data)
################################################################################

# imports and setup
!pip install rpy2==3.5.1 prettytable

import itertools
import time
import torch
import numpy as np
import pandas as pd
import os
import math

from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_curve, auc

# rpy2 for calling r's simulate_experiment_dataset()
%load_ext rpy2.ipython
from rpy2.robjects import pandas2ri
pandas2ri.activate()

# mount drive
from google.colab import drive
drive.mount('/content/drive')

# append your local bitimelygpt path
import sys
sys.path.append('/content/drive/MyDrive/BiTimelyGPT-main/BiTimelyGPT')

# import your modules
from data.data_pipeline import custom_collate
from models.BiTimelyGPT import BiTimelyGPT
from layers.optimization import get_linear_schedule_with_warmup, AdamW
from layers.heads import PretrainHead, ClfHead

set_seed = 42
torch.manual_seed(set_seed)
np.random.seed(set_seed)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.7/201.7 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rpy2: filename=rpy2-3.5.1-cp311-cp311-linux_x86_64.whl size=314977 sha256=d239c5a31702f9ad6c950fe2ba508e1bf6de2d462cb85d30c9d212d6f106bbba
  Stored in directory: /root/.cache/pip/wheels/e9/55/d1/47be85a5f3f1e1f4d1e91cb5e3a4dcb40dd72147f184c5a5ef
Successfully built rpy2
  Attempting uninstall: rpy2
    Found existing installation: rpy2 3.5.17
    Uninstalling rpy2-3.5.17:
      Successfully uninstalled rpy2-3.5.17
Mounted at /content/drive


In [ ]:
################################################################################
# define the r dgp script path
################################################################################

%%R
source("drive/MyDrive/sim_dgp.R")

In [ ]:
################################################################################
# a specialized pipeline for sim data
################################################################################

class SimulatedDataset(torch.utils.data.Dataset):
    """ each item is (sequence_tensor, label_tensor). """
    def __init__(self, samples, labels):
        super().__init__()
        self.samples = samples
        self.labels  = labels

    def __len__(self): self.samples.__len__()

    def __getitem__(self, idx):
        seq = self.samples[idx]
        lbl = self.labels[idx]
        return seq, lbl


def load_simulated_data(df_vars, df_outcomes, config):
    """ builds sequences + labels from dfs. """
    feature_map = {feat: i for i, feat in enumerate(config.feature_list)}

    samples = []
    labels = []

    subject_ids = df_vars["ID"].unique()
    for sid in subject_ids:
        sub_df  = df_vars[df_vars["ID"] == sid].copy()
        sub_out = df_outcomes[df_outcomes["ID"] == sid]

        # label
        if len(sub_out) == 1:
            label_val = sub_out["Y"].values[0]
        else:
            label_val = 0

        # sort by time
        sub_df = sub_df.sort_values("time")

        seq_tensors = []
        for t_val, group in sub_df.groupby("time"):
            feats = torch.zeros(len(config.feature_list), dtype=torch.float)
            for _, row in group.iterrows():
                var_name = row["variable"]
                if var_name in feature_map:
                    idx = feature_map[var_name]
                    feats[idx] = row["value"]
            seq_tensors.append(feats)

        if len(seq_tensors) == 0:
            seq_tensor = torch.zeros(1, len(config.feature_list))
        else:
            seq_tensor = torch.stack(seq_tensors, dim=0)

        samples.append(seq_tensor)
        labels.append(label_val)

    labels = torch.tensor(labels, dtype=torch.long)
    return samples, labels

In [ ]:
################################################################################
# define a configuration class for the model
################################################################################

class SimDataConfig:
    def __init__(self, p):
        # data characteristics
        self.feature_list = [f"X{i+1}" for i in range(p)]
        self.n_output = len(self.feature_list)
        self.n_clf_output = 2

        # common model hyperparameters
        self.num_layers = 6
        self.num_heads = 2
        self.d_model = 64
        self.qk_dim = 64
        self.v_dim = 128
        self.ffn_proj_size = 256
        self.d_ff = 128
        self.dropout = 0.1
        self.activation = 'gelu'

        # bitimelygpt specific
        self.use_bias_in_msr = False
        self.use_bias_in_mlp = True
        self.use_bias_in_msr_out = False
        self.use_default_gamma = False
        self.forward_impl = 'parallel'
        self.chunk_size = 12
        self.seq_len = 128

        # pretraining parameters
        self.pretrain_batch_size = 128
        self.pretrain_learning_rate = 1e-4
        self.pretrain_warmup_steps = 100

        # fine-tuning parameters
        self.finetune_batch_size = 32
        self.finetune_learning_rate = 2e-4
        self.finetune_epochs = 50
        self.finetune_warmup_steps = 100
        self.gradient_clip = 1.0

        # gpu and optimization settings
        self.use_gpu = torch.cuda.is_available()
        self.use_grad_ckp = False
        self.use_grad_accum = True
        self.accum_steps = 2
        self.use_amp = False
        self.use_multi_gpu = False
        self.devices = '0'

        self.output_retentions = False
        # self.head_type is set dynamically

In [ ]:
################################################################################
# utility: train & evaluate function with pretraining + fine-tuning
################################################################################
import rpy2.robjects as ro
from torch.utils.data import DataLoader, random_split
import math
from sklearn.metrics import roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import torch
import os
from torch import nn
from layers.optimization import get_linear_schedule_with_warmup, AdamW
from models.BiTimelyGPT import BiTimelyGPT
from layers.heads import ClfHead

def run_single_scenario(
    N, p,
    regularity,
    missing_prop,
    num_sync_groups,
    process_types,
    use_lags,
    block_missing=True,
    block_length=1.5,
    outcome_time=10,
    master_seed=42
):
    """
    
    """
    print(f"\n--- running scenario: n={N}, p={p}, reg={regularity}, miss={missing_prop}, sync={num_sync_groups}, lag={use_lags} ---")

    # ------------------ simulate in r ------------------
    ro.r.assign("N", N)
    ro.r.assign("p", p)
    ro.r.assign("regularity", regularity)
    ro.r.assign("missing_prop", missing_prop)
    ro.r.assign("num_sync_groups", num_sync_groups)
    if isinstance(process_types, list):
        ro.r.assign("process_types", ro.StrVector(process_types))
    else:
        ro.r.assign("process_types", ro.StrVector([process_types]*p))
    ro.r.assign("use_lags", use_lags)
    ro.r.assign("block_missing", block_missing)
    ro.r.assign("block_length", block_length)
    ro.r.assign("outcome_time", outcome_time)
    ro.r.assign("master_seed", master_seed)

    ro.r('''
        sim_dataset <- simulate_experiment_dataset(
            N               = N,
            p               = p,
            freq_min        = 0.5,
            freq_max        = 3.0,
            regularity      = regularity,
            num_sync_groups = num_sync_groups,
            process_types   = process_types,
            missing_prop    = missing_prop,
            block_missing   = block_missing,
            block_length    = block_length,
            outcome_time    = outcome_time,
            beta            = c(0, seq(-1, 1, length.out = 30)),
            use_lags        = use_lags,
            lag_hours       = ifelse(use_lags, 2, 0),
            master_seed     = master_seed
        )
    ''')

    df_vars_r     = ro.r('sim_dataset$vars_long')
    df_outcomes_r = ro.r('sim_dataset$outcome_df')

    # convert to pandas
    df_vars       = df_vars_r
    df_outcomes   = df_outcomes_r

    # ensure columns are correct types
    df_vars["ID"]     = df_vars["ID"].astype(int)
    df_vars["time"]   = df_vars["time"].astype(float)
    df_outcomes["ID"] = df_outcomes["ID"].astype(int)
    df_outcomes["time"]= df_outcomes["time"].astype(float)
    df_outcomes["Y"] = df_outcomes["Y"].astype(int)

    # track ground-truth distribution
    mean_true_prob = df_outcomes["true_prob"].mean()
    prop_positive  = df_outcomes["Y"].mean()
    print(f"data generated: {len(df_outcomes)} subjects, {prop_positive*100:.1f}% positive cases.")


    # ------------------ create train/val/test splits (70/15/15) ------------------
    config = SimDataConfig(p)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"using device: {device}")

    samples, labels = load_simulated_data(df_vars, df_outcomes, config)
    full_dataset    = SimulatedDataset(samples, labels)

    N_total = len(full_dataset)
    if N_total < 10:
         print(f"warning: very small dataset (n={N_total}). skipping scenario.")
         return { "test_loss": np.nan, "auc": np.nan, "accuracy": np.nan, "precision": np.nan,
                  "recall": np.nan, "f1": np.nan, "mean_true_prob": mean_true_prob, "prop_positive": prop_positive }

    N_train = int(0.70 * N_total)
    N_val   = int(0.15 * N_total)
    N_test  = N_total - N_train - N_val

    # adjust if calculated sizes are zero
    if N_train <= 0 or N_val <= 0 or N_test <= 0:
        print(f"warning: insufficient data for standard 70/15/15 split (n={N_total}). attempting adjusted split.")
        N_train = max(1, N_total - 2)
        N_val = 1 if N_total > 1 else 0
        N_test = 1 if N_total > 2 else 0
        if N_train <= 0 or (N_val + N_test == 0):
             print(f"error: cannot create a valid train/val/test split for n={N_total}. skipping scenario.")
             return { "test_loss": np.nan, "auc": np.nan, "accuracy": np.nan, "precision": np.nan,
                      "recall": np.nan, "f1": np.nan, "mean_true_prob": mean_true_prob, "prop_positive": prop_positive }

    print(f"splitting data: train={N_train}, val={N_val}, test={N_test}")
    generator = torch.Generator().manual_seed(master_seed)
    train_dataset, val_test_dataset = random_split(full_dataset, [N_train, N_val + N_test], generator=generator)

    # split validation and test sets
    if N_val + N_test > 0 :
        val_dataset, test_dataset = random_split(val_test_dataset, [N_val, N_test], generator=generator)
    else:
        val_dataset = torch.utils.data.Subset(val_test_dataset, [])
        test_dataset = torch.utils.data.Subset(val_test_dataset, [])


    # create dataloaders
    if len(train_loader) == 0:
        print("error: training loader is empty. cannot proceed. skipping scenario.")
        return { "test_loss": np.nan, "auc": np.nan, "accuracy": np.nan, "precision": np.nan,
                 "recall": np.nan, "f1": np.nan, "mean_true_prob": mean_true_prob, "prop_positive": prop_positive }


    # ------------------ initialize model for pretraining ------------------
    model = BiTimelyGPT(configs=config, head_type='pretrain').to(device)
    model.head_type = 'pretrain'
    print(f"initialized model with pretrain head. parameters: {sum(p.numel() for p in model.parameters())}")


    # ------------------ pretraining phase ------------------
    if N <= 500:
        config.pretrain_epochs = 15
    elif N <= 5000:
        config.pretrain_epochs = 15
    else:
        config.pretrain_epochs = 5
    print(f"--- starting pretraining phase ({config.pretrain_epochs} epochs based on n={N}) ---")

    pretrain_optimizer = AdamW(
        model.parameters(), lr=config.pretrain_learning_rate, weight_decay=0.01
    )
    num_pretraining_steps = len(train_loader) * config.pretrain_epochs
    pretrain_scheduler = get_linear_schedule_with_warmup(
        pretrain_optimizer, config.pretrain_warmup_steps, num_pretraining_steps
    )

    for epoch in range(config.pretrain_epochs):
        model.train()
        pretrain_losses = []
        pretrain_optimizer.zero_grad()

        for i, (batch_x, _, attention_mask) in enumerate(train_loader):
            batch_x, attention_mask = batch_x.to(device), attention_mask.to(device)

            loss = model(batch_x, y=None, retention_mask=None,
                         forward_impl=config.forward_impl, chunk_size=config.chunk_size)

            if torch.isnan(loss):
                print(f"warning: nan loss at pretrain epoch {epoch+1}, batch {i}. skipping update.")
                continue

            scaled_loss = loss / config.accum_steps
            scaled_loss.backward()

            if (i + 1) % config.accum_steps == 0 or (i + 1) == len(train_loader):
                pretrain_optimizer.step()
                pretrain_scheduler.step()
                pretrain_optimizer.zero_grad()

            pretrain_losses.append(loss.item())

        avg_pretrain_loss = np.mean(pretrain_losses) if pretrain_losses else np.nan
        print(f'pretrain epoch {epoch+1}/{config.pretrain_epochs} -- avg loss: {avg_pretrain_loss:.4f}')

    print("--- pretraining phase complete ---")

    # ------------------ 5) switch to fine-tuning ------------------
    print(f"--- switching to fine-tuning phase ({config.finetune_epochs} epochs) ---")
    model.head = ClfHead(config.d_model, config.n_clf_output).to(device)
    model.head_type = 'clf'
    print(f"model head switched to clfhead. total params now: {sum(p.numel() for p in model.parameters())}")

    optimizer = AdamW(model.parameters(), lr=config.finetune_learning_rate, weight_decay=0.01)
    num_finetune_steps = len(train_loader) * config.finetune_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, config.finetune_warmup_steps, num_finetune_steps
    )
    criterion = nn.CrossEntropyLoss()

    best_val_auc = -1.0
    patience = 10
    epochs_no_improve = 0
    warmup_epochs = 2
    best_model_path = f"best_model_temp_{master_seed}.pt"

    print("--- starting fine-tuning training loop ---")
    for epoch in range(config.finetune_epochs):
        # --- train (fine-tuning) ---
        model.train()
        train_losses = []
        optimizer.zero_grad()

        for i, (batch_x, batch_y, attention_mask) in enumerate(train_loader):
            batch_x, batch_y, attention_mask = (
                batch_x.to(device), batch_y.to(device), attention_mask.to(device)
            )

            assert model.head_type == 'clf', "model head_type should be 'clf'"
            hidden_states = model.conv_subsampling(batch_x)[0]
            hidden_states = model.input_projection(hidden_states)
            if attention_mask.shape[1] > hidden_states.shape[1]:
                subsampling_factor = batch_x.shape[1] // hidden_states.shape[1]
                attention_mask = attention_mask[:, ::subsampling_factor]
                attention_mask = attention_mask[:, :hidden_states.shape[1]]

            for block in model.blocks:
                out = block(hidden_states, retention_mask=attention_mask,
                            forward_impl=config.forward_impl, chunk_size=config.chunk_size)
                hidden_states = out[0]

            X = model.ln_f(hidden_states)
            logits = model.head(X)

            loss = criterion(logits, batch_y)
            scaled_loss = loss / config.accum_steps
            scaled_loss.backward()

            if (i + 1) % config.accum_steps == 0 or (i + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            train_losses.append(loss.item())

        avg_train_loss = np.mean(train_losses) if train_losses else np.nan

        # --- validation (fine-tuning) ---
        if len(val_loader) == 0:
            print(f"[fine-tune epoch {epoch+1}/{config.finetune_epochs}] trainloss={avg_train_loss:.4f}, skipping validation (empty loader)")
            if (epoch + 1) % 10 == 0:
                 torch.save(model.state_dict(), best_model_path)
                 print(f"saved model state at epoch {epoch+1} (no validation set).")
            continue

        model.eval()
        val_probs = []
        val_labels = []
        val_losses = []

        with torch.no_grad():
            for (batch_x, batch_y, attention_mask) in val_loader:
                batch_x, batch_y, attention_mask = (
                    batch_x.to(device), batch_y.to(device), attention_mask.to(device)
                )
                hidden_states = model.conv_subsampling(batch_x)[0]
                hidden_states = model.input_projection(hidden_states)
                if attention_mask.shape[1] > hidden_states.shape[1]:
                    subsampling_factor = batch_x.shape[1] // hidden_states.shape[1]
                    attention_mask = attention_mask[:, ::subsampling_factor]
                    attention_mask = attention_mask[:, :hidden_states.shape[1]]
                for block in model.blocks:
                    out = block(hidden_states, retention_mask=attention_mask,
                                forward_impl=config.forward_impl, chunk_size=config.chunk_size)
                    hidden_states = out[0]
                X_val = model.ln_f(hidden_states)
                logits = model.head(X_val)

                loss_val = criterion(logits, batch_y)
                val_losses.append(loss_val.item())
                probs = torch.softmax(logits, dim=1)[:, 1]
                val_probs.extend(probs.cpu().numpy())
                val_labels.extend(batch_y.cpu().numpy())

        avg_val_loss = np.mean(val_losses) if val_losses else np.nan
        val_auc = np.nan
        if len(np.unique(val_labels)) > 1:
            try:
                fpr_val, tpr_val, _ = roc_curve(val_labels, val_probs)
                val_auc = auc(fpr_val, tpr_val)
            except Exception as e:
                print(f"warning: could not calculate validation auc: {e}")
        else:
             val_auc = 0.0
             print(f"validation auc calculation skipped: only one class present ({np.unique(val_labels)}).")

        print(f"[fine-tune epoch {epoch+1}/{config.finetune_epochs}] "
              f"trainloss={avg_train_loss:.4f}, valloss={avg_val_loss:.4f}, valauc={val_auc:.4f}")

        if epoch + 1 > warmup_epochs and not np.isnan(val_auc):
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                epochs_no_improve = 0
                torch.save(model.state_dict(), best_model_path)
                print(f"==> val auc improved to {best_val_auc:.4f}. fine-tuned model saved.")
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    print(f"early stopping fine-tuning (no val auc improvement in {patience} epochs).")
                    break

    # ------------------ load best fine-tuned model, evaluate on test ------------------
    print("--- evaluating best fine-tuned model on test set ---")
    if len(test_loader) == 0:
        print("skipping test set evaluation (empty loader).")
        results_dict = { "test_loss": np.nan, "auc": np.nan, "accuracy": np.nan, "precision": np.nan,
                         "recall": np.nan, "f1": np.nan, "mean_true_prob": mean_true_prob, "prop_positive": prop_positive }
        if os.path.exists(best_model_path):
            os.remove(best_model_path)
        return results_dict


    if os.path.exists(best_model_path):
        model.load_state_dict(torch.load(best_model_path))
        print(f"loaded best fine-tuned model from {best_model_path}")
    else:
        print("warning: no best fine-tuned model saved. using the last model state for testing.")
        model.eval()

    all_probs = []
    all_labels = []
    test_losses = []

    with torch.no_grad():
        for (batch_x, batch_y, attention_mask) in test_loader:
            batch_x, batch_y, attention_mask = (
                batch_x.to(device), batch_y.to(device), attention_mask.to(device)
            )
            hidden_states = model.conv_subsampling(batch_x)[0]
            hidden_states = model.input_projection(hidden_states)
            if attention_mask.shape[1] > hidden_states.shape[1]:
                 subsampling_factor = batch_x.shape[1] // hidden_states.shape[1]
                 attention_mask = attention_mask[:, ::subsampling_factor]
                 attention_mask = attention_mask[:, :hidden_states.shape[1]]
            for block in model.blocks:
                out = block(hidden_states, retention_mask=attention_mask,
                            forward_impl=config.forward_impl, chunk_size=config.chunk_size)
                hidden_states = out[0]
            X_test = model.ln_f(hidden_states)
            logits = model.head(X_test)

            loss_test = criterion(logits, batch_y)
            test_losses.append(loss_test.item())
            probs = torch.softmax(logits, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    avg_test_loss = float(np.mean(test_losses)) if test_losses else 0.0
    test_auc = 0.0
    test_acc = 0.0
    test_prec = 0.0
    test_rec = 0.0
    test_f1 = 0.0

    if all_labels and all_probs:
        try:
            fpr_test, tpr_test, _ = roc_curve(all_labels, all_probs)
            test_auc = auc(fpr_test, tpr_test)

            threshold = 0.5
            preds = (np.array(all_probs) >= threshold).astype(int)
            labels_arr = np.array(all_labels)

            tp = np.sum((preds == 1) & (labels_arr == 1))
            fp = np.sum((preds == 1) & (labels_arr == 0))
            fn = np.sum((preds == 0) & (labels_arr == 1))
            tn = np.sum((preds == 0) & (labels_arr == 0))

            test_prec = tp / (tp + fp + 1e-9)
            test_rec = tp / (tp + fn + 1e-9)
            test_f1 = 2 * test_prec * test_rec / (test_prec + test_rec + 1e-9)
            test_acc = (tp + tn) / len(labels_arr) if len(labels_arr) > 0 else 0.0

        except ValueError as e:
            print(f"could not calculate test metrics: {e}")
        except Exception as e:
             print(f"an unexpected error occurred during metric calculation: {e}")

    else:
        print("skipping test metric calculation: no predictions or labels found.")

    print(f"test results: loss={avg_test_loss:.4f}, auc={test_auc:.4f}, f1={test_f1:.4f}")

    results_dict = {
        "test_loss":       avg_test_loss if not np.isnan(avg_test_loss) else 0.0,
        "auc":             test_auc,
        "accuracy":        test_acc,
        "precision":       test_prec,
        "recall":          test_rec,
        "f1":              test_f1,
        "mean_true_prob":  float(mean_true_prob) if not np.isnan(mean_true_prob) else 0.0,
        "prop_positive":   float(prop_positive) if not np.isnan(prop_positive) else 0.0
    }

    if os.path.exists(best_model_path):
        os.remove(best_model_path)

    return results_dict

In [ ]:
################################################################################
# main loop: full factorial over the 6 factors
################################################################################

# factors and levels:
Ns = [200, 1000, 10000]              # sample size
regularities = [0.2, 0.8]           # regularity
missing_props = [0.1, 0.4]          # missing data
sync_pattern_values = [1, "p"]      # sync
process_type_values = ["homogeneous", "mixed"]   # process types
outcome_dep_values  = ["direct", "lagged"]       # outcome dependency

# collect scenario combos
scenario_grid = list(itertools.product(
    Ns,
    regularities,
    missing_props,
    sync_pattern_values,
    process_type_values,
    outcome_dep_values
))

all_results = []
overall_start = time.time()

for scenario in scenario_grid:
    (N,
     regularity,
     missing_prop,
     sync_pattern,
     process_type_flag,
     outcome_dep) = scenario

    p = 30
    if process_type_flag == "homogeneous":
        process_types = ["ar1"]*p
    else:
        repeats = math.ceil(p/3)
        big_array = (["ar1","rw","seasonal"] * repeats)[:p]
        process_types = big_array

    if sync_pattern == "p":
        num_sync_groups = max(1, int(round(p / 10)))
    else:
        num_sync_groups = 1

    if outcome_dep == "direct":
        use_lags = False
    else:
        use_lags = True

    print("===================================================================")
    print(f"scenario => n={N}, reg={regularity}, missing={missing_prop}, "
          f"sync={sync_pattern}, process={process_type_flag}, outcome={outcome_dep}")
    print("===================================================================")

    start_time = time.time()
    results_dict = run_single_scenario(
        N=N,
        p=p,
        regularity=regularity,
        missing_prop=missing_prop,
        num_sync_groups=num_sync_groups,
        process_types=process_types,
        use_lags=use_lags,
        block_missing=True,
        block_length=1.5,
        outcome_time=10,
        master_seed=123
    )
    runtime = time.time() - start_time

    scenario_result = {
        "N":            N,
        "regularity":   regularity,
        "missing_prop": missing_prop,
        "sync_pattern": sync_pattern,
        "process_type": process_type_flag,
        "outcome_dep":  outcome_dep,
        "test_loss":    results_dict["test_loss"],
        "auc":          results_dict["auc"],
        "accuracy":     results_dict["accuracy"],
        "precision":    results_dict["precision"],
        "recall":       results_dict["recall"],
        "f1":           results_dict["f1"],

        "runtime_sec":  runtime,
        "mean_true_prob": results_dict["mean_true_prob"],
        "prop_positive":  results_dict["prop_positive"]
    }
    all_results.append(scenario_result)

overall_runtime = time.time() - overall_start
print(f"\nall scenarios completed in {overall_runtime/60:.2f} minutes.\n")

################################################################################
# convert results to a dataframe and save
################################################################################
df_results = pd.DataFrame(all_results)
df_results.sort_values(by=["N","regularity","missing_prop"], inplace=True)
print(df_results)

df_results.to_csv("drive/MyDrive/bitimelygpt_factorial_results_REP0.csv", index=False)
print("\nresults saved to bitimelygpt_factorial_results_REP0.csv")

Streaming output truncated to the last 5000 lines.
[Fine-tune Epoch 5/50] TrainLoss=0.7049, ValLoss=0.6847, ValAUC=0.5579
[Fine-tune Epoch 6/50] TrainLoss=0.6961, ValLoss=0.6743, ValAUC=0.5574
[Fine-tune Epoch 7/50] TrainLoss=0.7041, ValLoss=0.6741, ValAUC=0.5611
[Fine-tune Epoch 8/50] TrainLoss=0.7008, ValLoss=0.6778, ValAUC=0.5600
[Fine-tune Epoch 9/50] TrainLoss=0.6954, ValLoss=0.6925, ValAUC=0.5644
[Fine-tune Epoch 10/50] TrainLoss=0.6934, ValLoss=0.7093, ValAUC=0.5528
[Fine-tune Epoch 11/50] TrainLoss=0.6968, ValLoss=0.7056, ValAUC=0.5677
[Fine-tune Epoch 12/50] TrainLoss=0.6938, ValLoss=0.6920, ValAUC=0.5502
[Fine-tune Epoch 13/50] TrainLoss=0.6923, ValLoss=0.6893, ValAUC=0.5248
Early stopping fine-tuning (no val AUC improvement in 10 epochs).
--- Evaluating best fine-tuned model on Test Set ---
Loaded best fine-tuned model from best_model_temp_123.pt
Test Results: Loss=0.7613, AUC=0.4992, F1=0.0000
Scenario => N=1000, reg=0.2, missing=0.1, sync=p, process=homogeneous, outcome=la